In [25]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import fetch_openml

from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer, TVAESynthesizer

In [44]:
appointments = fetch_openml(name='Medical-Appointment-No-Shows', version=1, as_frame=True)

df = appointments.frame

#print(df.columns.tolist())

train, test = train_test_split(df, test_size=0.2)

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(train)
# Convert 'M' to 0 and 'F' to 1
df['Gender'] = df['Gender'].map({'M': 0, 'F': 1})
#metadata.update_column('No-show', sdtype='categorical')

synthesizer = CTGANSynthesizer(metadata, epochs=20)
synthesizer.fit(train)

C:\Users\emilo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sdv\single_table\base.py:183: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\emilo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


In [45]:
synthetic = synthesizer.sample(num_rows=len(train))

In [46]:
x_train, y_train = train.drop('No-show', axis=1), train['No-show']
x_test, y_teest = test.drop('No-show', axis=1), test['No-show']
x_syn, y_syn = test.drop('No-show', axis=1), synthetic['No-show']

In [47]:
combined = pd.concat([train, synthetic], ignore_index=True)
x_combined, y_combined = combined.drop('No-show', axis=1), combined['No-show']

In [48]:
baseline = RandomForestClassifier()
baseline.fit(x_train, y_train)

ValueError: could not convert string to float: 'F'

In [ ]:
baseline.score(x_test,y_test)

In [ ]:
combined_model = RandomForestClassifier()
combined_model.fit(x_combined, y_combined)
combined_model.score(x_test, y_test)

In [ ]:
synthetic_model = RandomForestClassifier()
synthetic_model.fit(x_syn, y_syn)
synthetic_mode.score(x_test, y_test)

In [ ]:
train.head()

In [ ]:
synthetic.head()

In [ ]:
train.describe()

In [ ]:
synthetic.describe()

In [30]:
print(df.columns.tolist())

['AppointmentID', 'Gender', 'ScheduledDay', 'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'No-show']
